In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard"
)

RAW_CPI = PROJECT_ROOT / "data" / "raw" / "cpi"
RAW_WPI = PROJECT_ROOT / "data" / "raw" / "wpi"
RAW_PPI = PROJECT_ROOT / "data" / "raw" / "ppi"

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed folder:", PROCESSED)

Project root: C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard
Processed folder: C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\data\processed


In [3]:
# ============================================================
# HISTORICAL CPI
# ============================================================

cpi_hist = pd.read_csv(
    RAW_CPI / r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\cpi_combined_historical_2012base.csv"
)

cpi_hist["date"] = pd.to_datetime(
    cpi_hist["year"].astype(str)
    + "-"
    + cpi_hist["month_code"].astype(str)
    + "-01"
)

cpi_hist = (
    cpi_hist
    .sort_values("date")
    .reset_index(drop=True)
)

# Calculate CPI inflation ourselves
cpi_hist["cpi_yoy"] = (
    cpi_hist["index"]
    .pct_change(12)
    * 100
)

cpi_hist = cpi_hist[
    ["date", "index", "cpi_yoy"]
].rename(
    columns={
        "index": "cpi_combined"
    }
)

print("Historical CPI:")
print(cpi_hist.shape)
print(cpi_hist["date"].min(), "→", cpi_hist["date"].max())

display(cpi_hist.head())
display(cpi_hist.tail())

Historical CPI:
(156, 3)
2013-01-01 00:00:00 → 2025-12-01 00:00:00


,date,cpi_combined,cpi_yoy
0,2013-01-01,104.6,NaN
1,2013-02-01,105.3,NaN
2,2013-03-01,105.5,NaN
3,2013-04-01,106.1,NaN
4,2013-05-01,106.9,NaN


,date,cpi_combined,cpi_yoy
151,2025-08-01,197.0,2.072539
152,2025-09-01,197.0,1.441813
153,2025-10-01,197.3,0.254065
154,2025-11-01,197.9,0.712468
155,2025-12-01,198.0,1.330604


In [5]:
# ============================================================
# WPI
# ============================================================

wpi = pd.read_excel(
    RAW_WPI / r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\wpi_monthly_2022_23base.xlsx",
    sheet_name="Sheet3",
    engine="openpyxl"
)

wpi_targets = [
    "All Commodities",
    "I PRIMARY ARTICLES",
    "II FUEL & POWER",
    "III. MANUFACTURED PRODUCTS"
]

wpi_core = wpi[
    wpi["Commodity Name"].isin(wpi_targets)
].copy()

wpi_month_columns = wpi.columns[4:]

wpi_long = wpi_core.melt(
    id_vars=[
        "Level",
        "Commodity Name",
        "Commodity Code",
        "Commodity Weight"
    ],
    value_vars=wpi_month_columns,
    var_name="month",
    value_name="index"
)

wpi_long["date"] = pd.to_datetime(
    wpi_long["month"],
    format="%b-%y"
)

wpi_long["series"] = wpi_long["Commodity Name"].map({
    "All Commodities": "wpi_all",
    "I PRIMARY ARTICLES": "wpi_primary",
    "II FUEL & POWER": "wpi_fuel",
    "III. MANUFACTURED PRODUCTS": "wpi_manufacturing"
})

wpi_master = (
    wpi_long
    .pivot(
        index="date",
        columns="series",
        values="index"
    )
    .reset_index()
)

wpi_master.columns.name = None

# WPI YoY
for col in [
    "wpi_all",
    "wpi_primary",
    "wpi_fuel",
    "wpi_manufacturing"
]:
    wpi_master[f"{col}_yoy"] = (
        wpi_master[col]
        .pct_change(12)
        * 100
    )

wpi_master = (
    wpi_master
    .sort_values("date")
    .reset_index(drop=True)
)

print("WPI:")
print(wpi_master.shape)
print(wpi_master["date"].min(), "→", wpi_master["date"].max())

display(wpi_master.head())

WPI:
(40, 9)
2023-04-01 00:00:00 → 2026-07-01 00:00:00


,date,wpi_all,wpi_fuel,wpi_manufacturing,wpi_primary,wpi_all_yoy,wpi_primary_yoy,wpi_fuel_yoy,wpi_manufacturing_yoy
0,2023-04-01,99.0,94.0,99.1,102.0,NaN,NaN,NaN,NaN
1,2023-05-01,98.6,91.4,98.9,102.3,NaN,NaN,NaN,NaN
2,2023-06-01,98.3,90.5,98.2,103.3,NaN,NaN,NaN,NaN
3,2023-07-01,99.1,90.6,97.8,107.9,NaN,NaN,NaN,NaN
4,2023-08-01,99.5,91.8,98.1,108.0,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# OUTPUT PPI
# ============================================================

ppi = pd.read_excel(
    RAW_PPI / r"C:\Users\Adity\OneDrive\Desktop\RESEARCH\Project Work\Inflation-Forecasting-Dashboard\output_ppi_monthly_2022_23base.xlsx",
    sheet_name="Sheet2",
    engine="openpyxl"
)

ppi_all = ppi[
    ppi["Commodity Name"]
    .astype(str)
    .str.fullmatch(
        "ALL COMMODITIES",
        case=False,
        na=False
    )
].copy()

ppi_month_columns = ppi.columns[4:]

ppi_master = pd.DataFrame({
    "date": pd.to_datetime(
        ppi_month_columns,
        format="%b-%y"
    ),
    "output_ppi": pd.to_numeric(
        ppi_all.iloc[0][ppi_month_columns].values,
        errors="coerce"
    )
})

ppi_master = (
    ppi_master
    .sort_values("date")
    .reset_index(drop=True)
)

ppi_master["output_ppi_yoy"] = (
    ppi_master["output_ppi"]
    .pct_change(12)
    * 100
)

print("Output PPI:")
print(ppi_master.shape)
print(ppi_master["date"].min(), "→", ppi_master["date"].max())

display(ppi_master.head())

Output PPI:
(40, 3)
2023-04-01 00:00:00 → 2026-07-01 00:00:00


,date,output_ppi,output_ppi_yoy
0,2023-04-01,99.1,NaN
1,2023-05-01,98.6,NaN
2,2023-06-01,98.2,NaN
3,2023-07-01,99.1,NaN
4,2023-08-01,99.4,NaN


In [7]:
# ============================================================
# CORE COMBINED DATASET
# ============================================================

master = (
    cpi_hist
    .merge(wpi_master, on="date", how="outer")
    .merge(ppi_master, on="date", how="outer")
    .sort_values("date")
    .reset_index(drop=True)
)

print("Master shape:", master.shape)
print("Duplicate dates:", master["date"].duplicated().sum())

display(master.head())
display(master.tail())

Master shape: (163, 13)
Duplicate dates: 0


,date,cpi_combined,cpi_yoy,wpi_all,wpi_fuel,wpi_manufacturing,wpi_primary,wpi_all_yoy,wpi_primary_yoy,wpi_fuel_yoy,wpi_manufacturing_yoy,output_ppi,output_ppi_yoy
0,2013-01-01,104.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-02-01,105.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-03-01,105.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-04-01,106.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-05-01,106.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,date,cpi_combined,cpi_yoy,wpi_all,wpi_fuel,wpi_manufacturing,wpi_primary,wpi_all_yoy,wpi_primary_yoy,wpi_fuel_yoy,wpi_manufacturing_yoy,output_ppi,output_ppi_yoy
158,2026-03-01,NaN,NaN,104.6,93.5,104.7,111.4,3.976143,2.578269,3.200883,4.804805,104.9,4.170804
159,2026-04-01,NaN,NaN,108.9,111.0,107.0,112.8,8.358209,3.963134,25.000000,6.679960,108.8,8.258706
160,2026-05-01,NaN,NaN,110.1,113.4,108.0,114.0,9.880240,5.263158,30.795848,7.676969,109.9,9.680639
161,2026-06-01,NaN,NaN,110.2,111.1,107.8,116.1,9.870389,7.004608,27.408257,7.477567,109.9,9.571286
162,2026-07-01,NaN,NaN,110.0,105.4,108.4,117.2,9.780439,8.518519,20.045558,8.291708,109.9,9.571286


In [8]:
print(
    "Duplicate columns:",
    master.columns[
        master.columns.duplicated()
    ].tolist()
)

Duplicate columns: []


In [9]:
cpi_long_model = cpi_hist.copy()

print(
    "CPI long model:",
    cpi_long_model["date"].min(),
    "→",
    cpi_long_model["date"].max()
)

print(
    "Observations:",
    len(cpi_long_model)
)

CPI long model: 2013-01-01 00:00:00 → 2025-12-01 00:00:00
Observations: 156


In [10]:
cpi_long_model.to_csv(
    PROCESSED / "cpi_long_model.csv",
    index=False
)

In [11]:
cpi_wpi_common_model = master[
    master["cpi_yoy"].notna()
    & master["wpi_all_yoy"].notna()
].copy()

print(
    "CPI + WPI:",
    cpi_wpi_common_model["date"].min(),
    "→",
    cpi_wpi_common_model["date"].max()
)

print(
    "Observations:",
    len(cpi_wpi_common_model)
)

CPI + WPI: 2024-04-01 00:00:00 → 2025-12-01 00:00:00
Observations: 21


In [12]:
cpi_wpi_common_model.to_csv(
    PROCESSED / "cpi_wpi_common_model.csv",
    index=False
)

In [13]:
cpi_wpi_ppi_common_model = master[
    master["cpi_yoy"].notna()
    & master["wpi_all_yoy"].notna()
    & master["output_ppi_yoy"].notna()
].copy()

print(
    "CPI + WPI + PPI:",
    cpi_wpi_ppi_common_model["date"].min(),
    "→",
    cpi_wpi_ppi_common_model["date"].max()
)

print(
    "Observations:",
    len(cpi_wpi_ppi_common_model)
)

CPI + WPI + PPI: 2024-04-01 00:00:00 → 2025-12-01 00:00:00
Observations: 21


In [14]:
cpi_wpi_ppi_common_model.to_csv(
    PROCESSED / "cpi_wpi_ppi_common_model.csv",
    index=False
)

In [15]:
print("========== FINAL DATASET CHECK ==========")

for name, df in {
    "CPI LONG": cpi_long_model,
    "CPI + WPI": cpi_wpi_common_model,
    "CPI + WPI + PPI": cpi_wpi_ppi_common_model
}.items():

    print("\n", name)
    print("Rows:", len(df))
    print("Start:", df["date"].min())
    print("End:", df["date"].max())
    print("Duplicate dates:", df["date"].duplicated().sum())

print("\n=========================================")

========== FINAL DATASET CHECK ==========

 CPI LONG
Rows: 156
Start: 2013-01-01 00:00:00
End: 2025-12-01 00:00:00
Duplicate dates: 0

 CPI + WPI
Rows: 21
Start: 2024-04-01 00:00:00
End: 2025-12-01 00:00:00
Duplicate dates: 0

 CPI + WPI + PPI
Rows: 21
Start: 2024-04-01 00:00:00
End: 2025-12-01 00:00:00
Duplicate dates: 0



In [16]:
for filename in [
    "cpi_long_model.csv",
    "cpi_wpi_common_model.csv",
    "cpi_wpi_ppi_common_model.csv"
]:
    path = PROCESSED / filename
    print(filename, "→", path.exists())

cpi_long_model.csv → True
cpi_wpi_common_model.csv → True
cpi_wpi_ppi_common_model.csv → True
